# Task 5 - Create a Judge Model

In this notebook, I build an automated LLM judge that evaluates generated product descriptions using the same rubric defined in Task 1.

The judge evaluates:
- Fluency
- Grammar
- Tone
- Length
- Grounding

Cost and latency are excluded because they are measured programmatically.

In [1]:
import os
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

In [2]:
BASE_DIR = Path.cwd().parent
INPUT_PATH = BASE_DIR / "outputs" / "assignment_01.xlsx"

load_dotenv(BASE_DIR / ".env")

NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY")
if not NEBIUS_API_KEY:
    raise ValueError("Missing NEBIUS_API_KEY in environment variables.")

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=NEBIUS_API_KEY,
)

df = pd.read_excel(INPUT_PATH)
df.head()

,product_name,Product_attribute_list,material,warranty,user_prompt,generated_description,latency_ms,input_tokens,output_tokens,word_count,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,error
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,Write a product description based on the follo...,"""Get ready to experience the latest and greate...",1085.23,216,88,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Write a product description based on the follo...,"""Unlock unparalleled mobile photography with t...",604.66,219,90,66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Write a product description based on the follo...,"""Take your mobile experience to the next level...",704.38,217,115,87,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Write a product description based on the follo...,"Immerse yourself in rich, immersive sound with...",700.14,216,113,80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Write a product description based on the follo...,"""Experience unparalleled sound and comfort wit...",630.15,210,85,60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
JUDGE_MODEL = "google/gemma-2-9b-it-fast"

In [4]:
class CriterionResult(BaseModel):
    explanation: str
    verdict: Literal["good", "ok", "bad"]


class JudgeOutput(BaseModel):
    fluency: CriterionResult
    grammar: CriterionResult
    tone: CriterionResult
    length: CriterionResult
    grounding: CriterionResult

In [11]:
JUDGE_PROMPT = """
You are an expert evaluator of e-commerce product descriptions.

Your task is to evaluate a generated product description using exactly the same rubric used in manual evaluation.

Evaluate only these five criteria:
1. Fluency
2. Grammar
3. Tone
4. Length
5. Grounding

Rubric:

Fluency
- good: The description reads smoothly, flows naturally, and sounds like human-written marketing copy without awkward phrasing or repetition.
- ok: The description is understandable but contains minor awkward phrasing, slight repetition, or less natural flow.
- bad: The description is difficult to read, feels robotic, or contains broken phrasing and noticeable repetition.

Grammar
- good: No grammatical, spelling, or punctuation errors. Sentences are well-structured and follow standard English conventions.
- ok: Minor grammatical, spelling, or punctuation errors that do not significantly affect readability or understanding.
- bad: Frequent or severe grammatical, spelling, or punctuation errors that make the text difficult to read or understand.

Tone
- good: Uses a friendly, credible, and persuasive sales voice appropriate for an e-commerce product description.
- ok: Generally appropriate tone, but somewhat generic, neutral, or less engaging than expected for product marketing.
- bad: Tone is inappropriate for a product description, such as overly robotic, exaggerated, untrustworthy, or unrelated to a sales context.

Length
- good: Between 50 and 90 words (inclusive).
- ok: Between 40–49 words or 91–110 words.
- bad: Fewer than 40 words or more than 110 words.

Grounding
- good: All information in the description is directly supported by the provided input (name, attributes, material, warranty). No new facts or claims are introduced.
- ok: Mostly grounded in the provided input, but includes minor unsupported embellishments that do not significantly alter the meaning.
- bad: Includes unsupported or fabricated information that is not present in the provided input, or contradicts the original data.

Important instructions:
- Exclude latency and cost from your evaluation.
- Judge grounding strictly by comparing the generated description against the provided product data.
- Do not assume missing details.
- For each criterion, provide the explanation first and then the verdict.
- Be strict and consistent.
- Return ONLY valid JSON.
- Use EXACTLY these top-level keys:
  "fluency", "grammar", "tone", "length", "grounding"
- For EACH top-level key, return an object with EXACTLY these keys:
  "explanation", "verdict"
- The verdict must be exactly one of:
  "good", "ok", "bad"

Return JSON in exactly this format:

{
  "fluency": {"explanation": "...", "verdict": "good"},
  "grammar": {"explanation": "...", "verdict": "good"},
  "tone": {"explanation": "...", "verdict": "ok"},
  "length": {"explanation": "...", "verdict": "good"},
  "grounding": {"explanation": "...", "verdict": "ok"}
}
""".strip()

In [12]:
def build_judge_input(row: pd.Series) -> str:
    return f"""
Product data:
- product_name: {row['product_name']}
- Product_attribute_list: {row['Product_attribute_list']}
- material: {row['material']}
- warranty: {row['warranty']}

Generated description:
{row['generated_description']}
""".strip()

In [13]:
import json


def run_judge(row: pd.Series, model: str = JUDGE_MODEL) -> JudgeOutput:
    judge_input = build_judge_input(row)

    response = client.chat.completions.create(
        model=model,
        temperature=0.2,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": judge_input},
        ],
        response_format={"type": "json_object"},
    )

    content = response.choices[0].message.content
    parsed = json.loads(content)

    return JudgeOutput(**parsed)

In [14]:
sample_row = df.iloc[0]
judge_result = run_judge(sample_row)

judge_result

JudgeOutput(fluency=CriterionResult(explanation='The description reads smoothly and naturally, using appropriate marketing language.', verdict='good'), grammar=CriterionResult(explanation='The grammar is correct throughout the description.', verdict='good'), tone=CriterionResult(explanation='The tone is friendly, enthusiastic, and persuasive, suitable for an e-commerce product description. However, it could be slightly more engaging.', verdict='ok'), length=CriterionResult(explanation="The description is 81 words long, falling within the 'good' range.", verdict='good'), grounding=CriterionResult(explanation='All information in the description is directly supported by the provided product data. There are no unsupported claims or embellishments.', verdict='good'))

## Judge Model Selection

For the judge model, I used `google/gemma-2-9b-it-fast`, since Task 2 used a Llama-based generation model.

I chose Gemma as a different model family to avoid using the same model for both generation and evaluation, while still keeping cost and latency reasonable.

If this model had struggled to follow the rubric or return consistent structured outputs, I would have switched to a larger model.

In [15]:
def flatten_judge_result(result: JudgeOutput) -> dict:
    return {
        "judge_fluency_explanation": result.fluency.explanation,
        "judge_fluency_verdict": result.fluency.verdict,
        "judge_grammar_explanation": result.grammar.explanation,
        "judge_grammar_verdict": result.grammar.verdict,
        "judge_tone_explanation": result.tone.explanation,
        "judge_tone_verdict": result.tone.verdict,
        "judge_length_explanation": result.length.explanation,
        "judge_length_verdict": result.length.verdict,
        "judge_grounding_explanation": result.grounding.explanation,
        "judge_grounding_verdict": result.grounding.verdict,
    }

## Why Explanation Comes Before Verdict

Placing the explanation before the verdict encourages the model to reason about each criterion before committing to a label.

This helps reduce shallow classification, improves consistency, and makes the evaluation more transparent and easier to review.